In [0]:
import json
import os
from dotenv import load_dotenv
from confluent_kafka import Consumer, KafkaError
from datetime import datetime
from pyspark.sql import Row

load_dotenv()

batch_records = []
BATCH_SIZE = 50

connection_string = os.getenv("CONNECTION_STR")
eventhub_name = os.getenv("EVENTHUB_NAME")
bootstrap_servers = os.getenv("BOOTSTRAP_SERVERS")

conf = {
    'bootstrap.servers': bootstrap_servers, 
    'security.protocol': 'SASL_SSL',
    'sasl.mechanism': 'PLAIN',
    'sasl.username': '$ConnectionString',  
    'sasl.password': connection_string,    
    'group.id': '$Default',  
    'auto.offset.reset': 'earliest'
}

consumer = Consumer(conf)
topic = eventhub_name 
consumer.subscribe([topic])

print("Listening for data")

try:
    while True:
        msg = consumer.poll(timeout=1.0)
        
        if msg is None:
            continue
        if msg.error():
            if msg.error().code() == KafkaError._PARTITION_EOF:
                continue
            else:
                print(f"Consumer error: {msg.error()}")
                break

        try:
            raw_data = msg.value().decode('utf-8')
            payload = json.loads(raw_data)
            
            if "data" in payload:
                payload = payload["data"]
            
            event_type = payload.get("e")
            symbol = payload.get("s")
            
            kline = payload.get("k", {})
            close_price = kline.get("c")
            open_price = kline.get("o")
            interval = kline.get("i")

            bronze_record = {
                "symbol": symbol,
                "event_type": event_type,
                "interval": interval,
                "open": float(open_price) if open_price else 0.0,
                "close": float(close_price) if close_price else 0.0,
                "kline":kline, 
                "raw_json": raw_data,
                "ingested_at": datetime.now()
            }

            batch_records.append(bronze_record)

            if len(batch_records) >= BATCH_SIZE:
                df = spark.createDataFrame([Row(**r) for r in batch_records])

                df.write\
                    .format("delta")\
                    .mode("append")\
                    .option("mergeSchema", "true") \
                    .saveAsTable("crp_arb_dev.bronze.crypto_data_delta")

                batch_records = []
            
            print(f"[{symbol} - {interval}] Open: {open_price} | Close: {close_price}")
            
        except json.JSONDecodeError:
            print("Failed to decode JSON. Raw message:", msg.value())

except KeyboardInterrupt:
    print("Stopping consumer...")
finally:
    consumer.close()